# Linear programming
*Algorithms for Optimization, Master Program in Finance, University of Pavia. Notebook of Lectures 1 and 2.*

| Section | Topic | Lecture |
|---|---|---|
| 0 | Setup | |
| 1 | A first LP in Gurobi: the investment firm | 1 |
| 2 | Standard form and slack variables | 1 |
| 3 | Allocating wealth: data, indexed variables and loops | 1 |
| 4 | Exercise: the warehouse problem | 1 |
| 5 | Duality | 2 |
| 6 | Complementary slackness | 2 |
| 7 | Exercise: the dual of the warehouse problem | 2 |
| 8 | At home: the dual in matrix form | 2 |

Run the cells in order. Exercises are marked with ✏️: write your code in the cell that says `# YOUR CODE HERE`.

## 0. Setup
We use the solver **Gurobi** through its Python package `gurobipy`.

The data files are stored on GitHub: `pd.read_csv` can read them directly from the web.

In [ ]:
!pip install gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/AmbrogioMB/AlgOpt/main/data/"

## 1. A first LP in Gurobi: the investment firm
A small investment firm allocates funds (in thousands of euros) between investment A (return 3 per unit) and investment B (return 5 per unit):
$$
\begin{aligned}
\max \quad & 3x + 5y \\
\text{s.t.} \quad & 2x + y \leq 10 \\
& x + 3y \leq 12 \\
& x, y \geq 0.
\end{aligned}
$$
Every part of the model becomes one line of code: variables (`addVar`), objective (`setObjective`), constraints (`addConstr`). Then `optimize()` calls the algorithm.

In [ ]:
firm = gp.Model("Investment firm")

# Decision variables: lower bound 0 is the default
x = firm.addVar(name="x")
y = firm.addVar(name="y")

# Objective
firm.setObjective(3*x + 5*y, GRB.MAXIMIZE)

# Constraints (we keep them in variables to query them later)
c1 = firm.addConstr(2*x + y <= 10, name="budget1")
c2 = firm.addConstr(x + 3*y <= 12, name="budget2")

firm.optimize()

Gurobi prints a **log**: size of the model, iterations of the algorithm, and the final status.
The results are stored in **attributes**:
* `firm.Status`: `GRB.OPTIMAL`, `GRB.INFEASIBLE`, `GRB.UNBOUNDED`, ... always check it before reading the solution;
* `x.X`: value of the variable `x` in the optimal solution;
* `firm.ObjVal`: optimal value of the objective;
* `c1.Slack`: right-hand side minus left-hand side of the constraint `c1` (zero if the constraint is binding).

In [ ]:
if firm.Status == GRB.OPTIMAL:
    print(f"Optimal value: {firm.ObjVal}")
    print(f"x = {x.X}, y = {y.X}")
    print(f"slack of budget1 = {c1.Slack}, slack of budget2 = {c2.Slack}")

We find the same solution as with the graphical method: $x = 3.6$, $y = 2.8$, return $24.8$ (i.e., €24,800), and both constraints are binding.

### Exercise 1.1 ✏️ (Exercise 3 of Lecture 1)
Solve the problem again when the return of investment A is €1 per unit. What is the optimal solution? Which constraints are binding?

In [ ]:
# YOUR CODE HERE
# Hint: build a new model (with a new name) and print the slacks of the two constraints.

### Exercise 1.2 ✏️ (Exercise 2 of Lecture 1)
Write a 2-variable LP that is **unbounded** and one that is **infeasible**, solve them and print `model.Status`. Compare the value with `GRB.OPTIMAL` (2), `GRB.INFEASIBLE` (3), `GRB.INF_OR_UNBD` (4), `GRB.UNBOUNDED` (5).

*Tip:* sometimes Gurobi only says "infeasible or unbounded" (status 4). Setting the parameter `model.Params.DualReductions = 0` before `optimize()` gives a definite answer.

In [ ]:
# YOUR CODE HERE

## 2. Standard form and slack variables
Every LP can be written in standard form $\min c^T x$ s.t. $Ax = b$, $x \geq 0$. For the investment firm we add a nonnegative **slack variable** to each $\leq$ constraint:
$$
2x + y + s_1 = 10, \qquad x + 3y + s_2 = 12, \qquad x, y, s_1, s_2 \geq 0.
$$
The slack $s_1$ is the unused part of the first budget. We do not need to do this for Gurobi (it does it internally), but it is useful to see that the result is the same.

In [ ]:
firm_sf = gp.Model("Investment firm, standard form")
x_sf = firm_sf.addVar(name="x")
y_sf = firm_sf.addVar(name="y")
s1 = firm_sf.addVar(name="s1")
s2 = firm_sf.addVar(name="s2")

# max 3x + 5y  is the same as  min -3x - 5y
firm_sf.setObjective(-3*x_sf - 5*y_sf, GRB.MINIMIZE)
firm_sf.addConstr(2*x_sf + y_sf + s1 == 10, name="budget1")
firm_sf.addConstr(x_sf + 3*y_sf + s2 == 12, name="budget2")
firm_sf.optimize()

print(f"Optimal value of the min problem: {firm_sf.ObjVal}  (so the max is {-firm_sf.ObjVal})")
print(f"x = {x_sf.X}, y = {y_sf.X}, s1 = {s1.X}, s2 = {s2.X}")

## 3. Allocating wealth: data, indexed variables and loops
This is the third problem of the introduction of the course (module 3). We allocate our wealth among $n$ assets: $w_i$ is the fraction invested in asset $i$, $\mu_i$ its expected (annual) return and $u_i$ the maximum weight allowed:
$$
\max_w \; \sum_{i=1}^n \mu_i w_i \quad \text{s.t.} \quad \sum_{i=1}^n w_i = 1, \quad 0 \leq w_i \leq u_i.
$$
The data are in the file `assets.csv` (simulated data, the tickers are fictional). We read it with pandas, as in the first part of the course.

In [ ]:
assets = pd.read_csv(DATA_URL + "assets.csv")
assets

With many variables we do not write them one by one:
* `addVars(keys, ...)` creates one variable for each key (here: each ticker) and returns a dictionary-like object;
* `gp.quicksum(...)` builds a sum with a `for` inside, like Python's `sum`, but faster;
* bounds can be given as dictionaries.

We first move the columns we need into dictionaries indexed by the ticker.

In [ ]:
tickers = list(assets["ticker"])
mu = dict(zip(assets["ticker"], assets["exp_return"]))
u = dict(zip(assets["ticker"], assets["max_weight"]))
sector = dict(zip(assets["ticker"], assets["sector"]))
vol = dict(zip(assets["ticker"], assets["volatility"]))

alloc = gp.Model("Wealth allocation")
alloc.Params.OutputFlag = 0  # no log from now on
w = alloc.addVars(tickers, lb=0, ub=u, name="w")
alloc.setObjective(gp.quicksum(mu[i] * w[i] for i in tickers), GRB.MAXIMIZE)
alloc.addConstr(gp.quicksum(w[i] for i in tickers) == 1, name="budget")
alloc.optimize()


def show_portfolio(model, w):
    print(f"Expected return: {100 * model.ObjVal:.2f}%")
    for i in w:
        if w[i].X > 1e-6:
            print(f"  {i:8s} {sector[i]:12s} {100 * w[i].X:6.2f}%")


show_portfolio(alloc, w)

Compare with the list of assets sorted by expected return: the solution is the "greedy" one we guessed in class (best asset up to its limit, then the second best, and so on). It is a **vertex** of the feasible region.

In [ ]:
assets.sort_values("exp_return", ascending=False)[["ticker", "sector", "exp_return", "max_weight"]]

Now add **sector limits**: at most 30% in Tech and at most 30% in Banks. We add one constraint per sector with a loop. A model can be modified and solved again.

In [ ]:
for s in ["Tech", "Banks"]:
    alloc.addConstr(gp.quicksum(w[i] for i in tickers if sector[i] == s) <= 0.30, name=f"sector_{s}")
alloc.optimize()
show_portfolio(alloc, w)

### Exercise 3.1 ✏️
Add a **risk budget**: the weighted average volatility of the portfolio must be at most 15%,
$$
\sum_{i=1}^n \sigma_i w_i \leq 0.15,
$$
where $\sigma_i$ is the column `volatility` (dictionary `vol`). This is a linear constraint, and $\sum_i \sigma_i w_i$ is an upper bound on the true volatility of the portfolio (diversification can only help). Which assets enter and which leave the portfolio? How much expected return do we lose?

In [ ]:
# YOUR CODE HERE

### Exercise 3.2 ✏️ (optional)
Replace the two sector limits with a limit of 30% for **every** sector in the file. *Hint:* `assets["sector"].unique()` gives the list of sectors. Build a new model from scratch.

In [ ]:
# YOUR CODE HERE

## 4. Exercise ✏️: the warehouse problem
A company distributes goods from 3 warehouses to 4 stores. Transport costs per unit, supply of each warehouse and demand of each store are given below. Formulate and solve an LP that minimizes the total shipping cost.

* **Variables**: amount shipped from each warehouse to each store.
* **Objective**: minimize total shipping cost.
* **Constraints**: shipments from each warehouse must not exceed its supply; shipments to each store must meet its demand.

*Hint:* `addVars(warehouses, stores, name="ship")` creates one variable for each pair `(i, j)`.

In [ ]:
warehouses = ["W1", "W2", "W3"]
stores = ["S1", "S2", "S3", "S4"]

supply = {"W1": 100, "W2": 150, "W3": 120}
demand = {"S1": 80, "S2": 90, "S3": 100, "S4": 100}
cost = {("W1", "S1"): 4, ("W1", "S2"): 6, ("W1", "S3"): 8, ("W1", "S4"): 10,
        ("W2", "S1"): 3, ("W2", "S2"): 5, ("W2", "S3"): 6, ("W2", "S4"): 8,
        ("W3", "S1"): 7, ("W3", "S2"): 8, ("W3", "S3"): 5, ("W3", "S4"): 6}

In [ ]:
# YOUR CODE HERE
warehouse = gp.Model("Warehouse")

## 5. Duality
For every LP there is a **dual** LP that gives the best upper bound (for a max problem) on its optimal value. For the investment firm:
$$
\begin{array}{ll}
\textbf{Primal} & \textbf{Dual} \\
\max \; 3x + 5y & \min \; 10u + 12v \\
\;\; 2x + y \leq 10 & \;\; 2u + v \geq 3 \\
\;\; x + 3y \leq 12 & \;\; u + 3v \geq 5 \\
\;\; x, y \geq 0 & \;\; u, v \geq 0
\end{array}
$$
Let us solve the dual and compare the optimal values.

In [ ]:
dual = gp.Model("Dual of the investment firm")
uu = dual.addVar(name="u")
vv = dual.addVar(name="v")
dual.setObjective(10*uu + 12*vv, GRB.MINIMIZE)
dual.addConstr(2*uu + vv >= 3, name="for_x")
dual.addConstr(uu + 3*vv >= 5, name="for_y")
dual.optimize()

print(f"Dual optimal value:   {dual.ObjVal}")
print(f"Primal optimal value: {firm.ObjVal}")
print(f"u = {uu.X}, v = {vv.X}")

The two optimal values coincide (**strong duality**). Any feasible dual solution gives an **upper bound** on the primal value: for example $u = 0, v = 3$ is feasible for the dual and gives $36 \geq 24.8$ (**weak duality**).

We did not need to write the dual: Gurobi computes the optimal dual solution together with the primal one. The dual values are the attribute `Pi` of the primal constraints, and they are the **shadow prices** of the constraints.

In [ ]:
print(f"Pi of budget1 = {c1.Pi}, Pi of budget2 = {c2.Pi}")

## 6. Complementary slackness
At the optimum:
* if a primal constraint has positive slack, its dual variable is zero: $u \cdot s_1 = 0$, $v \cdot s_2 = 0$;
* if a primal variable is positive, the corresponding dual constraint is binding: $x \cdot (2u + v - 3) = 0$, $y \cdot (u + 3v - 5) = 0$.

In [ ]:
print("u * slack1 =", uu.X * c1.Slack)
print("v * slack2 =", vv.X * c2.Slack)
print("x * (2u + v - 3) =", x.X * (2*uu.X + vv.X - 3))
print("y * (u + 3v - 5) =", y.X * (uu.X + 3*vv.X - 5))

### Exercise 6.1 ✏️ (Exercises 1 and 2 of Lecture 2)
Consider the LP
$$
\begin{aligned}
\min \quad & 5x_1 + 12x_2 + 4x_3 \\
\text{s.t.} \quad & x_1 + 2x_2 + x_3 = 10 \\
& 2x_1 - x_2 + 3x_3 = 8 \\
& x_1, x_2, x_3 \geq 0.
\end{aligned}
$$
1. Write its dual on paper, and use complementary slackness to find the optimal dual solution, knowing that $x_2 > 0$ and $x_3 > 0$ at the optimum.
2. Solve the primal with Gurobi and check your answer with the attribute `Pi` of the two constraints.
3. Solve the dual with Gurobi and check that the optimal values coincide.

In [ ]:
# YOUR CODE HERE

## 7. Exercise ✏️: the dual of the warehouse problem
Write the dual of the warehouse problem of Section 4 and solve it. Check that its optimal value coincides with the primal one, and compare the dual solution with the attribute `Pi` of the primal constraints.

*Hint:* the primal is a **minimization** with $\leq$ constraints (supply) and $\geq$ constraints (demand). Use the table of Lecture 2 (read from right to left): a $\geq$ constraint of a min problem gives a dual variable $\geq 0$, a $\leq$ constraint gives a dual variable $\leq 0$, and each primal variable $x_{ij} \geq 0$ gives a dual constraint $\alpha_i + \beta_j \leq c_{ij}$.

What is the economic meaning of the dual variable of a store?

In [ ]:
# YOUR CODE HERE

## 8. At home: the dual in matrix form
For an LP $\max c^T x$ s.t. $Ax \leq b$, $x \geq 0$ the dual is $\min b^T y$ s.t. $A^T y \geq c$, $y \geq 0$. Gurobi gives us the matrix $A$ of a model (`getA()`, a sparse matrix that we convert to a NumPy array), the right-hand sides and the objective coefficients. With the matrix interface `addMVar` we can then write the dual in three lines.

In [ ]:
A = firm.getA().toarray()
b = np.array(firm.getAttr("RHS", firm.getConstrs()))
c = np.array(firm.getAttr("Obj", firm.getVars()))
print("A =\n", A, "\nb =", b, "\nc =", c)

dual_m = gp.Model("Dual in matrix form")
yv = dual_m.addMVar(len(b), lb=0, name="y")
dual_m.setObjective(b @ yv, GRB.MINIMIZE)
dual_m.addConstr(A.T @ yv >= c)
dual_m.optimize()
print("dual solution:", yv.X, "value:", dual_m.ObjVal)

This works because all the constraints of `firm` are $\leq$ and the problem is a maximization. As an exercise, write a function `dual_of(model)` that also handles $\geq$ and $=$ constraints (attribute `Sense` of the constraints: `'<'`, `'>'`, `'='`) using the table of Lecture 2, and test it on the warehouse problem.